<a href="https://colab.research.google.com/github/miokobayashii/Lec2026/blob/main/models/FHN%26Rulkov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#神経細胞数理モデル 可視化ワーク

# FHN model

### 1. 発火現象の確認

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. FHNモデルの微分方程式の定義
def fhn_model(v, w, I_ext, phi, a, b):
    # 電位 v の変化（3次関数の非線形項 + 外部入力電流）
    dvdt = v - (v**3) / 3 - w + I_ext
    # 回復変数 w の変化（ゆっくり追いかける直線的な動き）
    dwdt = phi * (v + a - b * w)
    return dvdt, dwdt

# 2. パラメータと初期値の設定（綺麗な反復発火＝リミットサイクルが起きる定番値） #パラメータを変えるとNullclineや時間波形がどのように変化するでしょうか。
phi = 0.08
a = 0.7
b = 0.8
I_ext = 0.4   # この値を変化させる

# 初期状態（静止膜電位付近）　#初期状態も変化させて現象を観察してみましょう。
v0 = -1.0
w0 =  0.0

# 3. 時間軸の設定（オイラー法用）
t_end = 200
dt = 0.01  # 微小時間ステップ
t_steps = np.arange(0, t_end, dt)

# データ格納用のリスト
v_history = [v0]
w_history = [w0]

# 4. オイラー法によるループ計算
v, w = v0, w0
for _ in t_steps[1:]:
    # 現在の傾き（微分値）を計算
    dvdt, dwdt = fhn_model(v, w, I_ext, phi, a, b)

    # オイラー法：次の値 ＝ 現在の値 ＋ 傾き × dt
    v_next = v + dvdt * dt
    w_next = w + dwdt * dt

    # 状態の更新
    v, w = v_next, w_next

    v_history.append(v)
    w_history.append(w)

# 5. 可視化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=100)
max_y_ax = max(max(v_history),max(w_history), v0, w0)

y_limit_upper = (max_y_ax + 0.1) * 1.4

# 視点①：時間波形（神経細胞のパルス発火）
ax1.plot(t_steps, v_history, label='Membrane Potential (v)', color='dodgerblue', lw=2.5)
ax1.plot(t_steps, w_history, label='Recovery Variable (w)', color='orange', lw=2, linestyle='--')
ax1.set_xlabel('Time')
ax1.set_ylabel('State Value')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right')
ax1.set_title('Figure 1: Neural Spiking Time Series (FHN)', fontweight='bold', y=-0.2)
ax1.set_ylim(-3,y_limit_upper)
# 視点②：相平面（リミットサイクル）
ax2.plot(v_history, w_history, color='purple', lw=2.5, label='Trajectory')
ax2.scatter(v0, w0, color='red', s=100, label='Initial State', zorder=5)

# ヌルクライン（$\frac{dv}{dt}=0, \frac{dw}{dt}=0$ の線）を重ね書きして幾何学構造を見せる
v_range = np.linspace(-2.5, 2.5, 200)
v_nullcline = v_range - (v_range**3)/3 + I_ext
w_nullcline = (v_range + a) / b
ax2.plot(v_range, v_nullcline, color='blue', alpha=0.4, linestyle=':', label='v-nullcline')
ax2.plot(v_range, w_nullcline, color='green', alpha=0.4, linestyle=':', label='w-nullcline')

ax2.set_xlabel('Membrane Potential (v)')
ax2.set_ylabel('Recovery Variable (w)')
ax2.set_xlim(-2.5, 2.5)
ax2.set_ylim(-3, 3)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper left')
ax2.set_title('Figure 2: Phase Plane & Limit Cycle (FHN)', fontweight='bold', y=-0.2)

plt.tight_layout()
plt.show()

### 2. 平衡点の値

In [ ]:
import numpy as np

# ----------------------------------------------------
# 1. モデルのパラメータ設定
# ----------------------------------------------------
phi = 0.08
a = 0.7
b = 0.8

I_ext = 0.4 ##

# ----------------------------------------------------
# 2. 3次方程式の係数を整理する
#    w = \phi*(v + a - bw) = 0 より、w = (v + a) / b を dv/dt = 0 に代入
#    v - v^3/3 - (v + a)/b + I_ext = 0
#    両辺に -3 をかけて、v^3 の係数をプラスに整える：
#    v^3 + 3*(1/b - 1)*v + 3*(a/b - I_ext) = 0
# ----------------------------------------------------
# v^3 + c2*v^2 + c1*v + c0 = 0 の形に合わせる
c3 = 1.0
c2 = 0.0
c1 = 3.0 * (1.0 / b - 1.0)
c0 = 3.0 * (a / b - I_ext)

# ----------------------------------------------------
# 3. np.roots を使ってすべての v の解（根）を計算
# ----------------------------------------------------
coefficients = [c3, c2, c1, c0]
v_roots = np.roots(coefficients)

# ----------------------------------------------------
# 4. 結果の表示（実数解のみを抽出して、対応する w を計算）
# ----------------------------------------------------
print("==== FitzHugh-Nagumoモデル 平衡点解析 ====")
print(f"パラメータ: a={a}, b={b}, I_ext={I_ext}\n")

fixed_points = []
count = 1


for v_val in v_roots:
    # 複素数解を排除し、実数解のみをターゲットにする
    if np.isreal(v_val):
        v_real = np.real(v_val)
        # 対応する w の値を直線の方程式から計算
        w_real = (v_real + a) / b

        fixed_points.append((v_real, w_real))
        print(f"平衡点 {count}:")
        print(f"  v* = {v_real: .6f}")
        print(f"  w* = {w_real: .6f}")
        count += 1

if not fixed_points:
    print("実数解（平衡点）が見つかりませんでした。")

### 3. 平衡点でのヤコビ行列の固有値と固有ベクトル

In [ ]:
#ヤコビ行列を求める
# v, w, a, b, phiを用いて記述 (使わないものもある)
dfdv =  ###
dfdw =  ###
dgdv =  ###
dgdw =  ###
J = [[dfdv, dfdw], [dgdv, dgdw]]

#固有値と固有ベクトルを求める
lambda_list, lambda_vector_list =  np.linalg.eig(J)

#表示
count = 1
for l in lambda_list:
  print(f"固有値{count}, {l: .6f}")
  count += 1

count = 1
for i in range(lambda_vector_list.shape[1]): # Iterate over columns to get eigenvectors
    eigenvector = lambda_vector_list[:, i]
    formatted_components = []
    for comp in eigenvector:
        if comp.imag == 0:
            formatted_components.append(f"{comp.real:.6f}")
        else:
            formatted_components.append(f"{comp.real:.6f}{comp.imag:+.6f}j")
    print(f"固有ベクトル{count}, [{', '.join(formatted_components)}]")
    count += 1

## 課題1



1.   FHN modelのプログラムをI_extを変化させて実行し、安定な平衡点とリミットサイクルを確認してください。その時設定したI_extを示してください。
2.   1. で設定したパラメータにおける平衡点 $(v^*, w^*)$を求め、ヤコビ行列の固有値に基づき、平衡点の安定性について議論してください。

回答：

# Rulkov model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =================================================================
# 1. パラメータ設定
# =================================================================
# 講義のシナリオに合わせて、以下のいずれかのコメントアウトを解除してください。


# 【パターンA】固定点
alpha = 1.0
sigma = -1.0
mu = 0.001

# 【パターンB】周期解
#alpha = 6.0
#sigma = 0.1
#mu = 0.001

# 【パターンC】バースト発火
#alpha =     ### 設定してください
#sigma = -1.00
#mu = 0.001

# 【パターンD】興奮性
#alpha =      ### 設定してください
#sigma = -1.00
#mu = 0.001


# =================================================================
# 2. シミュレーション条件
# =================================================================
steps = 10000  # 総ステップ数（時間軸）

# 配列の確保（初期値の設定）
x = np.zeros(steps)
y = np.zeros(steps)

# 初期状態（適当な静止状態からスタート）
x[0] = -1.0
y[0] = -2.0

# =================================================================
# 3. ルルコフ・モデルの反復計算（差分方程式のメインループ）
# =================================================================
for n in range(steps - 1):
    # 第1式：高速に動く膜電位（非線形な分数関数）
    x[n+1] = (alpha / (1.0 + x[n]**2)) + y[n]

    # 第2式：低速に動く回復変数
    y[n+1] = y[n] - mu * (x[n] - sigma)

# =================================================================
# 4. 結果の描画（マplotting）
# =================================================================
plt.figure(figsize=(12, 8))

# 上段：膜電位 x_n の時系列プロット（神経パルスの確認）
plt.subplot(2, 1, 1)
plt.plot(x, color='#1f77b4', linewidth=0.8, label=f'Membrane potential $x_n$')
plt.title(f'Rulkov Map Simulation ($\\alpha$={alpha}, $\\sigma$={sigma}, $\\mu$={mu})', fontsize=14)
plt.ylabel('Membrane Potential $x_n$', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xlim(9000, steps) ###ここの数字を変えると表示される横軸の範囲を変えることができます。
plt.legend(loc='upper right')

# 下段：相平面プロット (x_n, y_n) の軌跡（数理的な構造の可視化）
# 後半の安定したリミットサイクル（バースト軌跡）を綺麗に見せるため、過渡変化を飛ばしてプロット
plt.subplot(2, 1, 2)
plt.plot(x[2000:], y[2000:], color='#ff7f0e', linewidth=0.6, alpha=0.8, label='Phase Portrait')
plt.xlabel('Fast Variable $x_n$', fontsize=12)
plt.ylabel('Slow Variable $y_n$', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 課題2

1.   Rulkov modelでパラメータを変化させてそれぞれの振動現象を観察してください。
2.   FHNモデルとRulkovモデルにより、神経細胞の数理モデルをシミュレーションを通して学びました。以下の観点について、あなたの研究分野との関連性を踏まえて、あなたの考えを述べてください。   
観点：微分方程式によるモデルと差分方程式によるモデルのメリットデメリット (プログラミングや計算速度の視点、生物学的な視点から)




回答：

### 全体を通して考えたこと、疑問に思ったこと、コメントなどあれば書いてください。(成績には関係しません。)

回答：